##02-2. 지하철 승하차

* API로 받은 **원본 그대로** PostgreSQL 테이블에 만들고 저장 (파일로 저장하지 않음)
* 아래 셀은 앞의 셀이 먼저 실행되어 있어야 함
  * `conn`, `ensure_conn()`, `save_df()` : DB 연결, DB 저장 공통함수
  * `fetch_all_rows()`, `check_api()` : 서울 열린데이터광장 API 수집 함수
  * `SEOUL_API_KEY`, `START_DATE`, `END_DATE` : 인증키, 수집 기간
* 컬럼은 API 응답 컬럼명을 소문자로 바꿔 전부 TEXT로 저장
* 추정값은 원본 테이블에 섞지 않고 이후 별도 테이블로 만듦

###02-2-0. 공통 설정과 수집·저장 함수

데이터셋은 `RAW_SOURCES`에 한 줄씩 등록하고, 수집·저장은 하나의 함수(`load_source`)로 처리

| 이름 | 데이터셋 | 조건값 | 상태 |
|---|---|---|---|
| `card_time_monthly` | 서울시 월별·시간대별 승하차 (OA-12252, `CardSubwayTime`) | 월 `YYYYMM` | 확인됨 |
| `daily_total` | 서울시 지하철호선별 역별 승하차 인원 (OA-12914, 일별) | 일 `YYYYMMDD` | 서비스명 **미확인 후보** → 아래 확인 셀로 검증 |
| `metro_daily_hour` | 서울교통공사 일별·시간대별 승하차 (OA-12921) | 일 | OpenAPI가 아닌 **파일(CSV) 제공**으로 보임 → 서비스명이 없으면 실행 전에 멈춤 |

* `service` 확인 방법: 데이터셋 페이지 **OpenAPI 탭의 샘플url** `http://openapi.seoul.go.kr:8088/sample/json/서비스명/1/5/조건값`
* `freq`: `"D"` 일(YYYYMMDD) / `"M"` 월(YYYYMM) / `""` 조건값 없음(1회 전체 조회)

In [ ]:
# ============================================================
# [공통 1] 설정값 : 데이터셋별 서비스명 / 저장 테이블 / 조건값 단위
#   - service 가 비어 있으면 API 호출 전에 멈춤
# ============================================================
RAW_SOURCES = {
    "card_time_monthly": {                     # 서울시 월별·시간대별 승하차 (OA-12252)
        "service": "CardSubwayTime",
        "table":   "RAW_SUBWAY_CARD_TIME",
        "freq":    "M",
    },
    "daily_total": {                           # 서울시 지하철호선별 역별 승하차 인원, 일별 (OA-12914)
        "service": "CardSubwayStatsNew",       # ※ 미확인 후보: [일별 2] 확인 셀에서 검증할 것
        "table":   "RAW_SUBWAY_DAILY_TOTAL",
        "freq":    "D",
    },
    "metro_daily_hour": {                      # 서울교통공사 일별·시간대별 승하차 (OA-12921)
        "service": "",                         # ← OpenAPI 서비스명이 있을 때만 입력
        "table":   "RAW_SUBWAY_METRO_DAILY_HOUR",
        "freq":    "D",
    },
}
LOAD_LOG     = "SUBWAY_RAW_LOAD_LOG"           # 데이터셋 + 기간별 적재 이력 테이블
MAX_RETRIES  = 3                               # 통신 실패 시 재시도 횟수
PERIOD_FORMAT = {"D": "%Y%m%d", "M": "%Y%m", "": None}


def check_sources(names=None):
    """설정 확인: 서비스명이 비어 있거나 freq가 잘못되면 호출 전에 중단"""
    for name in (names or RAW_SOURCES):
        cfg = RAW_SOURCES[name]
        if not cfg["service"].strip():
            raise ValueError(
                f"[{name}] service가 비어 있습니다. 데이터셋 페이지 OpenAPI 탭의 "
                f"샘플url에서 서비스명을 복사해 RAW_SOURCES에 넣으세요.")
        if cfg["freq"] not in PERIOD_FORMAT:
            raise ValueError(f"[{name}] freq는 'D', 'M', '' 중 하나여야 합니다: {cfg['freq']!r}")


def check_source(name, date_text="2024-01-02"):
    """api 정상접속 확인 : 첫 5행만 조회해서 컬럼 / 기간 / 단위 확인"""
    check_sources([name])
    cfg = RAW_SOURCES[name]
    fmt = PERIOD_FORMAT[cfg["freq"]]
    param = pd.Timestamp(date_text).strftime(fmt) if fmt else ""
    return check_api(service=cfg["service"], date_parameter=param)


def table_exists(table_name):
    with conn.cursor() as cur:
        cur.execute("SELECT to_regclass(%s::text)", [table_name.lower()])
        exists = cur.fetchone()[0] is not None
    conn.rollback()                                 # 확인용 트랜잭션 정리
    return exists

In [ ]:
# ============================================================
# [공통 2] 기간별 수집 → 저장 함수
#   - 한 기간(하루/한 달) 수집이 끝난 뒤에만 저장, 데이터 + 이력을 한 트랜잭션으로 커밋
#   - 통신/JSON 응답 실패는 최대 3회 재시도, 인증키/API 오류는 그대로 중단
#   - 이력 status : DONE(저장 완료) / EMPTY(빈 응답, 저장 안 함)
# ============================================================
def ensure_load_log():
    ensure_conn()
    with conn.cursor() as cur:
        cur.execute(f"""CREATE TABLE IF NOT EXISTS {LOAD_LOG} (
            dataset   TEXT NOT NULL,
            period    TEXT NOT NULL,
            row_cnt   BIGINT,
            status    TEXT,
            loaded_at TIMESTAMP DEFAULT now(),
            PRIMARY KEY (dataset, period))""")
    conn.commit()


def get_load_log(dataset):
    """{기간: 상태}"""
    with conn.cursor() as cur:
        cur.execute(f"SELECT period, status FROM {LOAD_LOG} WHERE dataset = %s", [dataset])
        log = {r[0]: r[1] for r in cur.fetchall()}
    conn.rollback()
    return log


def set_load_log(dataset, period, row_cnt, status, commit=True):
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {LOAD_LOG} (dataset, period, row_cnt, status) VALUES (%s, %s, %s, %s)
            ON CONFLICT (dataset, period) DO UPDATE
            SET row_cnt = EXCLUDED.row_cnt, status = EXCLUDED.status, loaded_at = now()""",
            [dataset, period, row_cnt, status])
    if commit:
        conn.commit()


def fetch_period(service, param):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fetch_all_rows(service, param)
        except RuntimeError as error:
            if "행 요청 실패" not in str(error):
                raise
            print(f"[요청 실패] {attempt}/{MAX_RETRIES}회")
            if attempt == MAX_RETRIES:
                raise
            wait_seconds = attempt * 3
            print(f"{wait_seconds}초 후 {param or '전체'}를 다시 조회합니다.")
            time.sleep(wait_seconds)


def load_period(name, period, param):
    cfg = RAW_SOURCES[name]
    df_period = fetch_period(cfg["service"], param)

    ensure_conn()                                       # 수집하는 동안 끊겼으면 재접속
    try:
        if df_period.empty:
            # 빈 응답을 '승하차 0명'으로 확정하지 않고 기록만 남김
            set_load_log(name, period, 0, "EMPTY", commit=False)
        else:
            save_df(df_period, cfg["table"], conn, mode="INSERT", commit=False)
            set_load_log(name, period, len(df_period), "DONE", commit=False)
        conn.commit()                                   # 데이터 + 이력 동시 커밋
    except Exception:
        conn.rollback()
        raise
    return len(df_period)

In [ ]:
# ============================================================
# [공통 3] 기간 전체 반복 적재
#   - 완료(DONE) / 빈 응답(EMPTY)인 기간은 건너뜀 → 다시 실행하면 이어서 진행
#   - retry_empty=True 로 빈 응답이었던 기간만 다시 조회 가능
#   - max_periods 로 일부만 먼저 시험 가능 (예: 3)
# ============================================================
def load_source(name, max_periods=None, retry_empty=False, stop_on_error=True):
    check_sources([name])
    cfg = RAW_SOURCES[name]
    fmt = PERIOD_FORMAT[cfg["freq"]]
    ensure_conn()
    ensure_load_log()

    if fmt is None:                                     # 조건값 없는 데이터셋: 1회 전체 조회
        periods = [("ALL", "")]
    else:
        periods = [(p.strftime(fmt), p.strftime(fmt))
                   for p in pd.period_range(START_DATE, END_DATE, freq=cfg["freq"])]

    log = get_load_log(name)
    skip = {"DONE"} if retry_empty else {"DONE", "EMPTY"}
    todo = [(p, q) for p, q in periods if log.get(p) not in skip]
    if max_periods:
        todo = todo[:max_periods]

    n_done = sum(1 for p, _ in periods if log.get(p) == "DONE")
    print(f"[{name}] 전체 {len(periods)}개 / 완료 {n_done}개 / 이번 처리 {len(todo)}개")

    failed = {}
    for i, (period, param) in enumerate(todo, 1):       # ← 기간별 반복문
        print(f"\n[{name} {i}/{len(todo)}] {period}")
        t0 = time.time()
        try:
            n = load_period(name, period, param)
            print(f"   ✔ {n:,}행 / {time.time()-t0:,.0f}초")
        except Exception as e:
            failed[period] = str(e)
            print(f"   ⚠ {period} 실패: {e}")
            if stop_on_error:
                print("   → 중단합니다. 원인 해결 후 다시 실행하면 이 기간부터 이어서 진행합니다.")
                break
        time.sleep(0.2)

    print(f"\n[{name}] 처리 종료: 실패 {len(failed)}개", failed if failed else "")
    return failed

###02-2-1. 지하철 역 정보

서울시 역사마스터 정보 (`subwayStationMaster`) > 출처: 서울 열린데이터광장

* 역ID·역명·호선·위도·경도를 확인할 수 있는 원본 테이블 `RAW_SUBWAY_STATION_MASTER` 생성
* 환승역 묶기, 좌표 검증은 저장 이후 단계에서 진행

In [ ]:
# api 정상접속 확인 : 첫 5행만 조회
df_subway_master_check = check_api(service="subwayStationMaster")

In [ ]:
# ============================================================
# [역 정보 DB 저장] API 전체 수집 → RAW_SUBWAY_STATION_MASTER 생성 + 저장
# ============================================================
MASTER_TABLE = "RAW_SUBWAY_STATION_MASTER"

if table_exists(MASTER_TABLE):
    print(f"{MASTER_TABLE}: 이미 있는 테이블이라 저장하지 않습니다.")
    print("다시 넣으려면 DROP TABLE 후 이 셀을 다시 실행하세요.")
else:
    raw_subway_master = fetch_all_rows("subwayStationMaster")   # 1,000행씩 끝까지 수집

    print("역사마스터:", raw_subway_master.shape)
    print("원본 컬럼:", raw_subway_master.columns.tolist())
    display(raw_subway_master.head())

    if raw_subway_master.empty:
        raise RuntimeError("수집된 역사마스터 데이터가 없어 저장하지 않습니다.")

    save_df(raw_subway_master, MASTER_TABLE, conn)

###02-2-2. 월별 승하차 정보

교통카드 정산시스템 시간대별 승하차 (`CardSubwayTime`, OA-12252) > 출처: 서울 열린데이터광장

* 조건값은 사용월(YYYYMM). 월별로 API 호출 → `RAW_SUBWAY_CARD_TIME`에 저장
* 월별 진행 상황은 `SUBWAY_RAW_LOAD_LOG`에 기록 (`DONE` 완료 / `EMPTY` 빈 응답)

In [ ]:
# api 정상접속 확인 : 2021년 1월 첫 5행만 조회
df_subway_time_check = check_source("card_time_monthly", "2021-01-01")

In [ ]:
# 먼저 1개월만 시험 저장해서 테이블이 잘 만들어지는지 확인
failed = load_source("card_time_monthly", max_periods=1)

In [ ]:
# 전체 기간 저장
failed = load_source("card_time_monthly")

In [ ]:
# ============================================================
# [이전 버전 이력 이전] 이전에 SUBWAY_CARD_TIME_LOAD_LOG 로 월별 적재를 이미 실행했다면
#   그 기록을 새 이력 테이블로 옮겨서 이미 받은 달을 다시 받지 않도록 함 (없으면 건너뜀)
# ============================================================
OLD_LOG = "SUBWAY_CARD_TIME_LOAD_LOG"
if table_exists(OLD_LOG):
    ensure_load_log()
    with conn.cursor() as cur:
        cur.execute(f"""
            INSERT INTO {LOAD_LOG} (dataset, period, row_cnt, status, loaded_at)
            SELECT 'card_time_monthly', use_month, row_cnt, status, loaded_at FROM {OLD_LOG}
            ON CONFLICT (dataset, period) DO NOTHING""")
        print("이전 이력 이전:", cur.rowcount, "건")
    conn.commit()
else:
    print("이전 버전 이력 테이블이 없어 건너뜁니다.")

###02-2-3. 일별 승하차 정보

월별 자료만으로는 일별 분석이 안 되어 일별 자료를 원본 그대로 추가 저장

1. 서울시 지하철호선별 역별 승하차 인원, 일별 (OA-12914) > 추정값 보정용 실제 일별 총량 (서울교통공사·코레일·공항철도·9호선)
2. 서울교통공사 일별·시간대별 승하차 (OA-12921) > 날짜·요일에 따른 시간대별 이용 패턴 파악용
   * 파일(CSV)로만 제공되는 경우 API 대신 파일을 읽어 같은 방식으로 DB에 저장해야 함 (서비스명이 비어 있으면 실행되지 않음)

In [ ]:
# api 정상접속 확인 : 첫 5행으로 컬럼 / 기간 / 역·노선 단위 확인
# 서비스명이 틀렸다면 여기서 ERROR 가 나므로 전체 수집 전에 알 수 있음
df_total_check = check_source("daily_total")

In [ ]:
# 먼저 3개 기간만 시험 저장해서 테이블이 잘 만들어지는지 확인
failed = load_source("daily_total", max_periods=3)

In [ ]:
# 전체 기간 저장 (약 2,000일: 끊겨도 다시 실행하면 이어서 진행)
failed = load_source("daily_total")

In [ ]:
# 서울교통공사 일별·시간대별 (OpenAPI 서비스명이 RAW_SOURCES에 입력된 경우에만 실행)
df_metro_check = check_source("metro_daily_hour")
failed = load_source("metro_daily_hour", max_periods=3)

###02-2-4. 저장 결과 확인

In [ ]:
# ============================================================
# 저장 결과 확인 : 기간, 건수, 컬럼, 역/노선 단위 파악
# ============================================================
conn.rollback()      # 오류로 중단된 트랜잭션이 남아 있으면 정리 (필수)

for name, cfg in RAW_SOURCES.items():
    table = cfg["table"].lower()
    print(f"\n===== {name} ({table}) =====")
    if not table_exists(cfg["table"]):
        print("테이블이 아직 없습니다.")
        continue
    with conn.cursor() as cur:
        cur.execute(f"SELECT status, COUNT(*), SUM(row_cnt) FROM {LOAD_LOG} "
                    "WHERE dataset = %s GROUP BY status ORDER BY status", [name])
        for status, n_periods, n_rows in cur.fetchall():
            print(f"{status}: {n_periods}개 기간 / {int(n_rows or 0):,}행")

        cur.execute(f"SELECT COALESCE(SUM(row_cnt), 0) FROM {LOAD_LOG} "
                    "WHERE dataset = %s AND status = 'DONE'", [name])
        log_rows = int(cur.fetchone()[0])
        cur.execute(f"SELECT COUNT(*) FROM {table}")
        real_rows = cur.fetchone()[0]
        print(f"실제 행 수: {real_rows:,}  →  로그와 {'일치 ✔' if real_rows == log_rows else '불일치 ⚠'}")

        cur.execute(f"SELECT * FROM {table} LIMIT 3")
        print("컬럼:", [d[0] for d in cur.description])
        for r in cur.fetchall():
            print(" ", r)
    conn.rollback()